In [9]:
import os
import json
import numpy as np
import pandas as pd
import torch

from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader


print("Imports successful.")

Imports successful.


In [10]:
print("=" * 60)
print("GPU CHECK")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA runtime:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print(
        f"GPU memory: "
        f"{props.total_memory / 1024**3:.2f} GB"
    )

    print(
        f"Currently allocated: "
        f"{torch.cuda.memory_allocated() / 1024**2:.2f} MB"
    )

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

GPU CHECK
PyTorch: 2.11.0+cu128
CUDA available: True
CUDA runtime: 12.8
GPU: NVIDIA GeForce RTX 2050
GPU memory: 4.00 GB
Currently allocated: 0.00 MB
Using device: cuda


In [11]:
GRAPH_DIR = "../data/processed/graph"

FEATURE_FILE = os.path.join(
    GRAPH_DIR,
    "node_features.npy"
)

EDGE_FILE = os.path.join(
    GRAPH_DIR,
    "edges_index.csv"
)

NODE_MAPPING_FILE = os.path.join(
    GRAPH_DIR,
    "node_mapping.csv"
)

print("Feature file:", FEATURE_FILE)
print("Edge file:", EDGE_FILE)
print("Node mapping:", NODE_MAPPING_FILE)

Feature file: ../data/processed/graph\node_features.npy
Edge file: ../data/processed/graph\edges_index.csv
Node mapping: ../data/processed/graph\node_mapping.csv


In [12]:
print("Loading node features...")

node_features = np.load(
    FEATURE_FILE
)

print(
    "Feature matrix shape:",
    node_features.shape
)

print(
    "Feature dtype:",
    node_features.dtype
)

print(
    "Contains NaN:",
    np.isnan(node_features).any()
)

print(
    "Contains Inf:",
    np.isinf(node_features).any()
)

Loading node features...
Feature matrix shape: (92081, 768)
Feature dtype: float32
Contains NaN: False
Contains Inf: False


In [13]:
print("Loading graph edges...")

edges = pd.read_csv(
    EDGE_FILE
)

print(
    "Edge columns:",
    list(edges.columns)
)

print(
    "Number of edges:",
    len(edges)
)

edges.head()

Loading graph edges...
Edge columns: ['source_id', 'target_id']
Number of edges: 206964


,source_id,target_id
0,0,57087
1,0,73381
2,0,21778
3,0,33043
4,0,33732


In [14]:
print("Loading graph edges...")

edges = pd.read_csv(
    EDGE_FILE
)

print(
    "Edge columns:",
    list(edges.columns)
)

print(
    "Number of edges:",
    len(edges)
)

edges.head()

Loading graph edges...
Edge columns: ['source_id', 'target_id']
Number of edges: 206964


,source_id,target_id
0,0,57087
1,0,73381
2,0,21778
3,0,33043
4,0,33732


In [17]:
print("Loading graph edges...")

edges = pd.read_csv(
    EDGE_FILE
)

print("Edge columns:")
print(edges.columns.tolist())

print("\nFirst 5 rows:")
display(edges.head())

Loading graph edges...
Edge columns:
['source_id', 'target_id']

First 5 rows:


,source_id,target_id
0,0,57087
1,0,73381
2,0,21778
3,0,33043
4,0,33732


In [19]:
print("Loading graph edges...")

edges = pd.read_csv(
    EDGE_FILE
)

print("Edge columns:")
print(edges.columns.tolist())

print("\nFirst 5 rows:")
display(edges.head())

Loading graph edges...
Edge columns:
['source_id', 'target_id']

First 5 rows:


,source_id,target_id
0,0,57087
1,0,73381
2,0,21778
3,0,33043
4,0,33732


In [21]:
print("Loading graph edges...")

edges = pd.read_csv(
    EDGE_FILE
)

print("Edge columns:")
print(edges.columns.tolist())

print("\nFirst 5 rows:")
display(edges.head())

Loading graph edges...
Edge columns:
['source_id', 'target_id']

First 5 rows:


,source_id,target_id
0,0,57087
1,0,73381
2,0,21778
3,0,33043
4,0,33732


In [22]:
print("=" * 60)
print("EDGE VALIDATION")
print("=" * 60)

print("Columns:", edges.columns.tolist())

# We expect exactly two columns containing node IDs.
if len(edges.columns) < 2:
    raise ValueError(
        "The edge file does not contain enough columns."
    )

# Show the actual column names so we can verify them
print("\nUsing:")
print("Source column:", edges.columns[0])
print("Target column:", edges.columns[1])

source = edges.iloc[:, 0].to_numpy()
target = edges.iloc[:, 1].to_numpy()

num_nodes = node_features.shape[0]

print("\nNumber of nodes:", num_nodes)
print("Number of edges:", len(edges))

print("Minimum source:", source.min())
print("Maximum source:", source.max())

print("Minimum target:", target.min())
print("Maximum target:", target.max())

invalid_source = (
    (source < 0) |
    (source >= num_nodes)
).sum()

invalid_target = (
    (target < 0) |
    (target >= num_nodes)
).sum()

self_loops = (
    source == target
).sum()

print("\nInvalid source IDs:", invalid_source)
print("Invalid target IDs:", invalid_target)
print("Self loops:", self_loops)

EDGE VALIDATION
Columns: ['source_id', 'target_id']

Using:
Source column: source_id
Target column: target_id

Number of nodes: 92081
Number of edges: 206964
Minimum source: 0
Maximum source: 92080
Minimum target: 0
Maximum target: 92080

Invalid source IDs: 0
Invalid target IDs: 0
Self loops: 0


In [23]:
print("Converting graph to PyTorch...")

x = torch.from_numpy(
    node_features
)

edge_index = torch.tensor(
    np.vstack([
        source,
        target
    ]),
    dtype=torch.long
)

data = Data(
    x=x,
    edge_index=edge_index
)

print(data)

Converting graph to PyTorch...
Data(x=[92081, 768], edge_index=[2, 206964])


In [24]:
data

Data(x=[92081, 768], edge_index=[2, 206964])

In [25]:
print("=" * 60)
print("NEIGHBOR SAMPLER TEST")
print("=" * 60)

loader = NeighborLoader(
    data,
    num_neighbors=[15, 10],
    batch_size=256,
    shuffle=True,
    num_workers=0
)

batch = next(iter(loader))

print("Batch loaded successfully.")

print("Batch object:")
print(batch)

print(
    "\nBatch nodes:",
    batch.x.shape[0]
)

print(
    "Batch features:",
    batch.x.shape
)

print(
    "Batch edges:",
    batch.edge_index.shape
)

print(
    "Seed nodes:",
    batch.batch_size
)

NEIGHBOR SAMPLER TEST
Batch loaded successfully.
Batch object:
Data(x=[3136, 768], edge_index=[2, 3097], n_id=[3136], e_id=[3097], num_sampled_nodes=[3], num_sampled_edges=[2], input_id=[256], batch_size=256)

Batch nodes: 3136
Batch features: torch.Size([3136, 768])
Batch edges: torch.Size([2, 3097])
Seed nodes: 256


In [26]:
num_neighbors=[15, 10]

In [27]:
batch_size=256

In [28]:
print("=" * 60)
print("GPU BATCH TEST")
print("=" * 60)

batch_gpu = batch.to(device)

print(
    "Batch device:",
    batch_gpu.x.device
)

print(
    "Feature shape:",
    batch_gpu.x.shape
)

print(
    "Edge shape:",
    batch_gpu.edge_index.shape
)

print(
    "GPU memory allocated:",
    f"{torch.cuda.memory_allocated() / 1024**2:.2f} MB"
)

GPU BATCH TEST
Batch device: cuda:0
Feature shape: torch.Size([3136, 768])
Edge shape: torch.Size([2, 3097])
GPU memory allocated: 9.28 MB
